[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openlayer-ai/openlayer-python/blob/main/examples/tracing/claude_agent_sdk/claude_agent_sdk_tracing.ipynb)

# Tracing the Claude Agent SDK with Openlayer

This notebook shows how to enable Openlayer tracing for applications built with Anthropic's [Claude Agent SDK](https://github.com/anthropics/claude-agent-sdk-python). After one line of setup, every `query()` becomes an Openlayer trace with nested steps for assistant turns, tool calls (built-in + MCP), subagents, session metadata, cost, and tokens.

Three scenarios, building up in complexity:

1. **Quickstart** — single `query()` with built-in tools (Read / Glob / Grep)
2. **MCP + subagent** — register an in-process MCP tool, dispatch a subagent
3. **Multi-stage orchestration** — wrap multiple `query()` calls inside one outer step so the whole pipeline is a single trace

## 1. Install dependencies

In [ ]:
!pip install openlayer 'claude-agent-sdk>=0.1.81'

## 2. Set environment variables

You need three secrets:

- `OPENLAYER_API_KEY` — get from [openlayer.com/settings/api-keys](https://app.openlayer.com/settings/api-keys)
- `OPENLAYER_INFERENCE_PIPELINE_ID` — the inference pipeline you want to stream traces to
- `ANTHROPIC_API_KEY` — your Anthropic API key

In [ ]:
import os

os.environ["OPENLAYER_API_KEY"] = "YOUR_OPENLAYER_API_KEY"
os.environ["OPENLAYER_INFERENCE_PIPELINE_ID"] = "YOUR_INFERENCE_PIPELINE_ID"
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

## 3. Enable tracing — one line

`trace_claude_agent_sdk()` monkey-patches `claude_agent_sdk.query` and `ClaudeSDKClient` so every subsequent call is auto-traced. It composes with any hooks you've configured yourself — your hooks are not replaced.

In [ ]:
from openlayer.lib import trace_claude_agent_sdk

trace_claude_agent_sdk()

## 4. Scenario 1 — quickstart

A simple `query()` with read-only built-in tools. The resulting trace contains one root `Claude Agent SDK query` AGENT step with nested `CHAT_COMPLETION` turns and `TOOL` calls.

In [ ]:
from claude_agent_sdk import ResultMessage, ClaudeAgentOptions, query


async def scenario_1():
    options = ClaudeAgentOptions(
        model="claude-haiku-4-5",
        allowed_tools=["Read", "Glob", "Grep"],
    )
    async for message in query(
        prompt="Find any .py files in the current directory and tell me roughly what they do.",
        options=options,
    ):
        if isinstance(message, ResultMessage):
            print(message.result)  # noqa: T201


await scenario_1()

## 5. Scenario 2 — in-process MCP tool + subagent

Register a custom MCP tool that counts files by extension, and dispatch a `code-reviewer` subagent. In the trace, the MCP call appears as a `TOOL` step with `metadata.mcp_server="file-stats"` and `metadata.mcp_tool_name="count_files"`. The subagent dispatch appears as a nested `AGENT` step (`Agent: code-reviewer`) containing the subagent's own assistant turns and tool calls.

In [ ]:
from pathlib import Path
from collections import Counter

from claude_agent_sdk import AgentDefinition, tool, create_sdk_mcp_server


@tool("count_files", "Count files in a directory grouped by extension", {"directory": str})
async def count_files(args):
    target = Path(args["directory"]).expanduser().resolve()
    if not target.is_dir():
        return {"content": [{"type": "text", "text": f"Not a directory: {target}"}], "isError": True}
    counts = Counter()
    for f in target.rglob("*"):
        if f.is_file():
            counts[f.suffix or "(no ext)"] += 1
    body = "\n".join(f"{ext}: {n}" for ext, n in counts.most_common(20))
    return {"content": [{"type": "text", "text": body or "(empty)"}]}


mcp_server = create_sdk_mcp_server("file-stats", "1.0.0", tools=[count_files])

code_reviewer = AgentDefinition(
    description="Briefly reviews a code file for clarity, correctness, and style.",
    prompt=(
        "You are a senior code reviewer. Read the file the user names, then return ONE "
        "specific observation about its quality. Two sentences max."
    ),
    tools=["Read", "Grep"],
    model="claude-haiku-4-5",
)


async def scenario_2():
    options = ClaudeAgentOptions(
        model="claude-haiku-4-5",
        system_prompt=(
            "You are a codebase explorer. Count files in the directory, then dispatch "
            "the code-reviewer subagent on ONE interesting file. Output a 2-line summary."
        ),
        # Subagent tools must also be in the session's allowed_tools.
        allowed_tools=["Glob", "Read", "Grep", "Agent", "mcp__file-stats__count_files"],
        mcp_servers={"file-stats": mcp_server},
        agents={"code-reviewer": code_reviewer},
        permission_mode="acceptEdits",
        max_turns=10,
    )
    async for message in query(
        prompt=f"Analyze the directory at: {Path.cwd()}",
        options=options,
    ):
        if isinstance(message, ResultMessage):
            print(message.result)  # noqa: T201


await scenario_2()

## 6. Scenario 3 — multi-stage orchestration

When you want multiple `query()` calls to appear as one trace, wrap them in `tracer.create_step()`. Each inner `query()` becomes a nested `AGENT` step under your outer step.

This example splits an audit workflow into two phases: an inventory query, then a review query that dispatches a specialist subagent. Both are children of one outer `codebase-audit` AGENT step.

In [ ]:
from openlayer.lib.tracing import tracer
from openlayer.lib.tracing.enums import StepType


async def phase_inventory():
    options = ClaudeAgentOptions(
        model="claude-haiku-4-5",
        system_prompt=(
            "Inventory the current working directory and pick ONE .py file. "
            "End your last message with: TARGET: <absolute path>"
        ),
        allowed_tools=["Glob", "Read", "mcp__file-stats__count_files"],
        mcp_servers={"file-stats": mcp_server},
        max_turns=6,
    )
    async for message in query(prompt=f"Working directory: {Path.cwd()}", options=options):
        if isinstance(message, ResultMessage):
            for line in reversed((message.result or "").splitlines()):
                if line.strip().startswith("TARGET:"):
                    return line.strip()[len("TARGET:"):].strip()
    return None


async def phase_review(target):
    options = ClaudeAgentOptions(
        model="claude-haiku-4-5",
        system_prompt="Dispatch code-reviewer on the file and return its observation verbatim.",
        allowed_tools=["Agent", "Read", "Grep"],
        agents={"code-reviewer": code_reviewer},
        permission_mode="acceptEdits",
        max_turns=6,
    )
    async for message in query(prompt=f"Review this file: {target}", options=options):
        if isinstance(message, ResultMessage):
            return message.result
    return None


with tracer.create_step(name="codebase-audit", step_type=StepType.AGENT) as outer:
    target = await phase_inventory()
    review = await phase_review(target) if target else None
    outer.output = review or "(no review produced)"
    outer.log(metadata={"audited_file": target})

print("audited:", target)  # noqa: T201
print("\nreview:\n", review)  # noqa: T201

## 7. What to look for in the Openlayer trace

Open your inference pipeline and click into each trace. You should see:

**Scenario 1** — a single root `AGENT` step (`Claude Agent SDK query`) with assistant turn(s) and tool calls as children.

**Scenario 2** — same root, plus a `TOOL` step for the MCP call (with `metadata.mcp_server` and `metadata.mcp_tool_name`) and a nested `AGENT` step named `Agent: code-reviewer` containing the subagent's own chat completions and tool steps.

**Scenario 3** — one outer `codebase-audit` AGENT step, with two nested `Claude Agent SDK query` AGENT steps inside it (one per phase), and the review phase contains its own `Agent: code-reviewer` nested step.

Click any `AGENT` step to see `system_prompt`, `agent_config`, `agents_defined`, `options`, and the raw `ResultMessage`. Click any `CHAT_COMPLETION` step for per-turn model, prompt/completion tokens, thinking content, and raw assistant message. Click any `TOOL` step for input, output, latency, and the originating `tool_use_id`.